# Telco Customer Churn — Exploratory Data Analysis

This notebook explores the IBM Telco Customer Churn dataset to understand:
1. Data quality and missing values
2. Churn rate by customer segments
3. Feature distributions and correlations
4. Key patterns that inform the ML agents

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/raw/telco_churn.csv')
print(f'Shape: {df.shape}')
print(f'\nColumn types:\n{df.dtypes}')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nTotalCharges blanks: {(df["TotalCharges"].str.strip() == "").sum()}')
df.head()

In [ ]:
# Clean TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn_binary'] = (df['Churn'] == 'Yes').astype(int)
df.describe()

## 2. Churn Rate Overview

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall churn rate
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=churn_counts.index, autopct='%1.1f%%',
            colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Overall Churn Distribution')

# Churn by contract type
churn_by_contract = df.groupby('Contract')['Churn_binary'].mean() * 100
churn_by_contract.plot(kind='bar', ax=axes[1], color=['#3498db', '#e67e22', '#2ecc71'])
axes[1].set_title('Churn Rate by Contract Type')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 3. Tenure & Charges Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Tenure distribution by churn
for churn_val in ['No', 'Yes']:
    subset = df[df['Churn'] == churn_val]
    axes[0].hist(subset['tenure'], bins=30, alpha=0.6, label=f'Churn={churn_val}')
axes[0].set_title('Tenure Distribution by Churn Status')
axes[0].set_xlabel('Tenure (months)')
axes[0].legend()

# Monthly charges by churn
sns.boxplot(data=df, x='Churn', y='MonthlyCharges', ax=axes[1],
            palette=['#2ecc71', '#e74c3c'])
axes[1].set_title('Monthly Charges by Churn Status')

# Scatter: tenure vs monthly charges
colors = df['Churn_binary'].map({0: '#2ecc71', 1: '#e74c3c'})
axes[2].scatter(df['tenure'], df['MonthlyCharges'], c=colors, alpha=0.3, s=10)
axes[2].set_title('Tenure vs Monthly Charges')
axes[2].set_xlabel('Tenure (months)')
axes[2].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

## 4. Service Feature Impact

In [ ]:
service_cols = ['PhoneService', 'MultipleLines', 'InternetService',
                'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(service_cols):
    churn_rate = df.groupby(col)['Churn_binary'].mean() * 100
    churn_rate.plot(kind='bar', ax=axes[i], color='#3498db')
    axes[i].set_title(f'Churn Rate by {col}')
    axes[i].set_ylabel('Churn %')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 5. Payment Method & Billing

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Churn by payment method
churn_by_payment = df.groupby('PaymentMethod')['Churn_binary'].mean() * 100
churn_by_payment.sort_values().plot(kind='barh', ax=axes[0], color='#e67e22')
axes[0].set_title('Churn Rate by Payment Method')
axes[0].set_xlabel('Churn Rate (%)')

# Churn by paperless billing
churn_by_billing = df.groupby('PaperlessBilling')['Churn_binary'].mean() * 100
churn_by_billing.plot(kind='bar', ax=axes[1], color=['#2ecc71', '#e74c3c'])
axes[1].set_title('Churn Rate by Paperless Billing')
axes[1].set_ylabel('Churn Rate (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
# Create numeric DataFrame
numeric_df = df[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn_binary']].copy()
numeric_df['Partner'] = (df['Partner'] == 'Yes').astype(int)
numeric_df['Dependents'] = (df['Dependents'] == 'Yes').astype(int)
numeric_df['PaperlessBilling'] = (df['PaperlessBilling'] == 'Yes').astype(int)
numeric_df['TechSupport'] = df['TechSupport'].map({'Yes': 1, 'No': 0, 'No internet service': 0})
numeric_df['OnlineSecurity'] = df['OnlineSecurity'].map({'Yes': 1, 'No': 0, 'No internet service': 0})
numeric_df['Contract_Monthly'] = (df['Contract'] == 'Month-to-month').astype(int)
numeric_df['Elec_Check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)
numeric_df['Fiber'] = (df['InternetService'] == 'Fiber optic').astype(int)

plt.figure(figsize=(12, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 7. Key Findings

### Churn Drivers (Correlational — Agent 3 tests causality)

1. **Contract type** is the strongest correlate: Month-to-month customers churn at ~42% vs ~3% for two-year contracts.
2. **Electronic check** payment has the highest churn (~45%) — suggests payment friction.
3. **Fiber optic** internet has higher churn than DSL — likely a price sensitivity effect.
4. **Tech Support and Online Security** are associated with lower churn when present.
5. **Tenure** is inversely related to churn — new customers (0-12 months) are most at risk.
6. **Monthly charges** are higher for churned customers — price is a factor.

### Implications for Agent Design

- **Agent 1**: Clusters should capture the contract × internet × tenure interaction.
- **Agent 2**: Payment-driven vs dissatisfaction-driven churn separation is well-supported.
- **Agent 3**: DML should reveal whether TechSupport/OnlineSecurity *cause* retention or just correlate.